# 02 — Patient-level landmark aggregation

This notebook converts the cleaned, pre-landmark visit records from Notebook 01 into one unimputed row per patient. It follows the revision aggregation structure wherever defensible and applies the approved feature-specific rules from the updated audit.

The primary output contains the locked 1,040-patient revision-aligned cohort and 26 candidate features. Outcomes and timing remain in a separate metadata file. The broader cohort and alternative representations are saved only as named sensitivities.

## How to use this notebook

Run every cell from top to bottom in a fresh kernel after Notebook 01 passes all checks. The notebook displays only aggregate summaries. It performs no imputation, statistical testing, feature engineering, feature selection, or model fitting.

## 1. Set up paths and libraries

Load the required libraries and locate the project root from any folder inside the repository.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

working_directory = Path.cwd().resolve()
project_root = next(
    (
        folder
        for folder in [working_directory, *working_directory.parents]
        if (folder / "AGENTS.md").is_file()
        and (folder / "docs/research_protocol.md").is_file()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError("Launch this notebook from within the project directory.")

print("Project root:", project_root)


Project root: /Users/rafsan_temp/Library/CloudStorage/OneDrive-SeattleUniversity/SU Projects/pd-fall-risk


Use Notebook 01 as the sole cleaned-data input. The aggregation rules and feature-set definitions are declared in this notebook, and all new files go to the final-run output directory.


In [2]:
cleaned_directory = (
    project_root
    / "results/final_pipeline/06_final_fit_and_performance_summary/01_data_cleaning/cleaned_visits"
)
output_directory = (
    project_root
    / "results/final_pipeline/06_final_fit_and_performance_summary/02_patient_level_aggregation"
)
output_directory.mkdir(parents=True, exist_ok=True)

data_directory = project_root / "data"

print("Cleaned inputs:", cleaned_directory.relative_to(project_root))
print("New outputs:", output_directory.relative_to(project_root))


Cleaned inputs: results/final_pipeline/06_final_fit_and_performance_summary/01_data_cleaning/cleaned_visits
New outputs: results/final_pipeline/06_final_fit_and_performance_summary/02_patient_level_aggregation


## 2. Load the cleaned visit tables

Declare the 15 cleaned sources produced by Notebook 01. Keeping this list explicit makes the aggregation input boundary easy to audit.

In [3]:
source_names = [
    "demographics",
    "family_history",
    "diagnosis_history",
    "dopamine_therapy",
    "moca",
    "freezing",
    "scopa_aut",
    "neuroqol_mobility",
    "gds15",
    "updrs_part_i",
    "updrs_part_i_patient",
    "updrs_part_iii",
    "updrs_part_iv",
    "other_clinical",
    "vital_signs",
]

missing_sources = [
    name
    for name in source_names
    if not (cleaned_directory / f"{name}_clean.csv").is_file()
]
assert not missing_sources, f"Missing cleaned inputs: {missing_sources}"


Load each table, parse its dates, and recheck that every source record is on or before the patient-specific landmark.

In [4]:
def load_clean(source_name):
    path = cleaned_directory / f"{source_name}_clean.csv"
    frame = pd.read_csv(
        path,
        dtype={"PATNO": "string"},
        low_memory=False,
    )
    frame["DATE"] = pd.to_datetime(frame["DATE"], errors="raise")
    frame["index_date"] = pd.to_datetime(frame["index_date"], errors="raise")
    assert frame["DATE"].le(frame["index_date"]).all(), source_name
    return frame


tables = {
    source_name: load_clean(source_name)
    for source_name in source_names
}

print(f"Loaded {len(tables)} landmark-eligible cleaned tables.")


Loaded 15 landmark-eligible cleaned tables.


Show the record and patient coverage of each input without displaying patient-level data.

In [5]:
input_summary = pd.DataFrame([
    {
        "table": source_name,
        "records": len(frame),
        "patients": frame["PATNO"].nunique(),
        "columns": frame.shape[1],
    }
    for source_name, frame in tables.items()
])

display(input_summary)


,table,records,patients,columns
0,demographics,1043,1043,16
1,family_history,1193,1021,8
2,diagnosis_history,1043,1043,13
3,dopamine_therapy,1245,838,8
4,moca,4984,1040,8
5,freezing,2944,977,8
6,scopa_aut,5421,1040,9
7,neuroqol_mobility,2944,976,15
8,gds15,5421,1040,24
9,updrs_part_i,9320,1040,14


## 3. Reconstruct the outcome-eligible patient spine

Retain the revision cohort definition and latest-outcome rule. Outcome data are reconstructed only to define patient eligibility, the landmark, and the separate modeling label file.

In [6]:
eligible_statuses = ["Enrolled", "Complete", "Withdraw Deceased"]

status = pd.read_csv(
    data_directory / "Participant_Status_24Aug2026.csv",
    dtype={"PATNO": "string"},
)
falls = pd.read_csv(
    data_directory / "Determination_of_Freezing_and_Falls_24Aug2026.csv",
    dtype={"PATNO": "string"},
)

eligible_ids = status.loc[
    status["COHORT"].eq(1)
    & status["ENROLL_STATUS"].isin(eligible_statuses),
    "PATNO",
]
falls = falls.loc[falls["PATNO"].isin(eligible_ids)].copy()
falls["outcome_date"] = pd.to_datetime(
    falls["INFODT"],
    format="%m/%Y",
    errors="coerce",
)


Select each patient's latest dated, observed `FLNFR12M`, map its ordinal response to the three outcome classes, and calculate the index month 12 months earlier.

In [7]:
observed_outcomes = falls.dropna(
    subset=["FLNFR12M", "outcome_date"]
).copy()
latest_month = observed_outcomes.groupby("PATNO")["outcome_date"].transform("max")
latest_outcomes = observed_outcomes.loc[
    observed_outcomes["outcome_date"].eq(latest_month)
].copy()

assert latest_outcomes["PATNO"].is_unique, (
    "A patient has multiple observed outcomes in the latest month."
)

metadata = latest_outcomes[["PATNO", "outcome_date", "FLNFR12M"]].rename(
    columns={"FLNFR12M": "falls_raw"}
)
metadata["falls_class"] = pd.to_numeric(metadata["falls_raw"]).map(
    lambda value: 0 if value == 0 else 1 if value == 1 else 2
)
metadata["index_date"] = metadata["outcome_date"] - pd.DateOffset(months=12)
metadata = metadata.sort_values("PATNO").reset_index(drop=True)

assert len(metadata) == 1_280
assert metadata["PATNO"].is_unique
print(f"Outcome-eligible patient spine: {len(metadata):,}")


Outcome-eligible patient spine: 1,280


## 4. Define deterministic aggregation helpers

Record same-month ties for every latest-value operation. A coherent latest assessment prefers the most complete row in the latest eligible month and uses `REC_ID` only as a reproducible final tie-breaker.

In [8]:
selection_audits = []


def audit_latest_month(frame, fields, label):
    usable = frame.loc[frame[fields].notna().any(axis=1)].copy()
    latest_date = usable.groupby("PATNO")["DATE"].transform("max")
    latest_rows = usable.loc[usable["DATE"].eq(latest_date)]
    counts = latest_rows.groupby("PATNO").size()
    tied_ids = counts[counts.gt(1)].index
    tied_rows = latest_rows.loc[latest_rows["PATNO"].isin(tied_ids)]

    if len(tied_rows):
        conflicts = (
            tied_rows.groupby("PATNO")[fields]
            .nunique(dropna=False)
            .gt(1)
            .any(axis=1)
            .sum()
        )
    else:
        conflicts = 0

    selection_audits.append({
        "selection": label,
        "patients_with_value": usable["PATNO"].nunique(),
        "latest_month_ties": len(tied_ids),
        "ties_with_different_values": int(conflicts),
    })


Select one coherent latest row for multi-field assessments. This prevents constructing a questionnaire or visit from fields recorded on different rows.

In [9]:
def latest_row(frame, fields, label):
    audit_latest_month(frame, fields, label)
    work = frame.loc[frame[fields].notna().any(axis=1)].copy()
    work["_complete"] = work[fields].notna().sum(axis=1)
    work["_rec"] = work["REC_ID"].astype("string").fillna("")
    work = work.sort_values(["PATNO", "DATE", "_complete", "_rec"])

    return (
        work.groupby("PATNO", as_index=False)
        .tail(1)
        .set_index("PATNO")[fields]
    )


Select latest values independently when each field represents its own current measurement. Each field still uses the same deterministic month and `REC_ID` rule.

In [10]:
def latest_values(frame, fields, label_prefix):
    pieces = []

    for field in fields:
        audit_latest_month(frame, [field], f"{label_prefix}: {field}")
        valid = frame.dropna(subset=[field]).copy()
        valid["_rec"] = valid["REC_ID"].astype("string").fillna("")
        valid = valid.sort_values(["PATNO", "DATE", "_rec"])
        pieces.append(valid.groupby("PATNO")[field].last())

    return pd.concat(pieces, axis=1)


Define historical-maximum, positive-dominant history, and safe column-joining helpers. These reproduce the approved revision-style aggregation categories.

In [11]:
def max_values(frame, fields, suffix=None):
    result = frame.groupby("PATNO")[fields].max()
    return result.add_suffix(suffix) if suffix else result


def categorical_history(frame, field):
    ordered = frame.copy()
    ordered["_rec"] = ordered["REC_ID"].astype("string").fillna("")
    ordered = ordered.sort_values(["PATNO", "DATE", "_rec"])

    def choose(group):
        values = group[field].dropna()
        if values.eq("Yes").any():
            return "Yes"
        return values.iloc[-1] if len(values) else pd.NA

    return (
        ordered.groupby("PATNO")
        .apply(choose, include_groups=False)
        .rename(field)
    )


def add_features(base, features):
    return base.join(features, how="left")


## 5. Aggregate landmark-derived and contextual predictors

Start with one row for every outcome-eligible patient. Calculate age at the landmark and retain one coherent eligible demographic record.

In [12]:
patient = pd.DataFrame(index=metadata["PATNO"]).rename_axis("PATNO")
index_dates = metadata.set_index("PATNO")["index_date"]

demographic_fields = [
    "BIRTHDT", "SEX", "RAASIAN", "RABLACK", "RAHAWOPI",
    "RAINDALS", "RANOS", "RAWHITE", "RAUNKNOWN",
]
demographics = latest_row(
    tables["demographics"],
    demographic_fields,
    "demographics coherent row",
)

demographics["BIRTHDT"] = pd.to_datetime(
    demographics["BIRTHDT"],
    errors="coerce",
)
demographic_landmark = index_dates.reindex(demographics.index)
demographics["Age"] = (
    demographic_landmark.dt.year
    - demographics["BIRTHDT"].dt.year
    - (
        demographic_landmark.dt.month < demographics["BIRTHDT"].dt.month
    ).astype("Int64")
)


Derive the single-label race representation used in the experimental manifest and join age, sex, and race without using outcomes.

In [13]:
race_map = {
    "RAASIAN": "Asian",
    "RABLACK": "Black or African American",
    "RAHAWOPI": "Native Hawaiian or Other Pacific Islander",
    "RAINDALS": "American Indian or Alaska Native",
    "RANOS": "Not Otherwise Specified",
    "RAWHITE": "White",
    "RAUNKNOWN": "Unknown",
}


def infer_race(row):
    marked = [
        label
        for field, label in race_map.items()
        if row.get(field) == 1
    ]
    if len(marked) == 1:
        return marked[0]
    if len(marked) > 1:
        return "Multiple"
    return pd.NA


demographics["Race"] = demographics.apply(infer_race, axis=1)
patient = add_features(
    patient,
    demographics[["Age", "SEX", "Race"]].rename(columns={"SEX": "Sex"}),
)


Use one coherent eligible diagnosis record, calculate disease duration at the landmark, and preserve the corrected `DXPOSINS` definition.

In [14]:
diagnosis_fields = [
    "PDDXDT", "DXTREMOR", "DXRIGID", "DXBRADY", "DXPOSINS", "DOMSIDE",
]
diagnosis = latest_row(
    tables["diagnosis_history"],
    diagnosis_fields,
    "diagnosis coherent row",
)
diagnosis["PDDXDT"] = pd.to_datetime(diagnosis["PDDXDT"], errors="coerce")
diagnosis["Years_since_PD_diagnosis"] = (
    index_dates.reindex(diagnosis.index) - diagnosis["PDDXDT"]
).dt.days / 365.25

patient = add_features(
    patient,
    diagnosis[[
        "Years_since_PD_diagnosis",
        "DXTREMOR",
        "DXRIGID",
        "DXBRADY",
        "DXPOSINS",
        "DOMSIDE",
    ]],
)


Aggregate family history and treatment initiation as eligible-history occurrence variables. A documented `Yes` is preserved; otherwise the latest eligible nonmissing category remains.

In [15]:
patient = add_features(
    patient,
    categorical_history(tables["family_history"], "ANYFAMPD").to_frame(),
)
patient = add_features(
    patient,
    categorical_history(tables["dopamine_therapy"], "DOPTHERST").to_frame(),
)


## 6. Aggregate recent status and historical burden

Use the latest valid value for cognition and complete depression totals. These variables are intended to describe status nearest the landmark.

In [16]:
patient = add_features(
    patient,
    latest_values(tables["moca"], ["MCATOT"], "latest status"),
)
patient = add_features(
    patient,
    latest_values(tables["gds15"], ["GDS_TOTAL"], "latest status"),
)


Use historical maxima for freezing and the two prespecified SCOPA-AUT symptoms, matching their approved prior-burden interpretation.

In [17]:
patient = add_features(
    patient,
    max_values(tables["freezing"], ["FRZGT12M"]),
)
patient = add_features(
    patient,
    max_values(tables["scopa_aut"], ["SCAU14", "SCAU16"]),
)


Aggregate the rater-completed and patient-reported Part I variables as historical maxima. Maximum eligible constipation is the locked primary representation.

In [18]:
part_i_rater = [
    "NP1RTOT", "NP1COG", "NP1HALL", "NP1DPRS",
    "NP1ANXS", "NP1APAT", "NP1DDS",
]
part_i_patient = [
    "NP1PTOT", "NP1SLPN", "NP1SLPD", "NP1PAIN",
    "NP1URIN", "NP1CNST", "NP1LTHD", "NP1FATG",
]

patient = add_features(
    patient,
    max_values(tables["updrs_part_i"], part_i_rater),
)
patient = add_features(
    patient,
    max_values(tables["updrs_part_i_patient"], part_i_patient),
)


Keep latest eligible constipation as a mutually exclusive sensitivity representation. It does not enter the primary table beside the maximum.

In [19]:
constipation_latest = latest_values(
    tables["updrs_part_i_patient"],
    ["NP1CNST"],
    "latest-status alternative",
).rename(columns={"NP1CNST": "NP1CNST_LATEST"})
patient = add_features(patient, constipation_latest)

constipation_comparison = patient[["NP1CNST", "NP1CNST_LATEST"]].dropna().copy()
constipation_comparison["difference"] = (
    constipation_comparison["NP1CNST"]
    - constipation_comparison["NP1CNST_LATEST"]
)

print("Patients with both representations:", len(constipation_comparison))
print(
    "Maximum greater than latest:",
    int(constipation_comparison["difference"].gt(0).sum()),
)


Patients with both representations: 1040
Maximum greater than latest: 381


Select one latest coherent Neuro-QoL form and calculate the preserved revision Gaussian composite before choosing the patient-level row.

In [20]:
neuroqol_fields = [
    "NQMOB37", "NQMOB30", "NQMOB26", "NQMOB32",
    "NQMOB25", "NQMOB33", "NQMOB31", "NQMOB28",
]
neuroqol = tables["neuroqol_mobility"].copy()
weighted = np.exp(
    -((neuroqol[neuroqol_fields] - 2.5) ** 2)
    / (2 * 1.2 ** 2)
)
neuroqol["NQ_GAUSSIAN_REVISION"] = weighted.mean(axis=1) * 8

neuroqol_output_fields = neuroqol_fields + ["NQ_GAUSSIAN_REVISION"]
patient = add_features(
    patient,
    latest_row(
        neuroqol,
        neuroqol_output_fields,
        "Neuro-QoL coherent form",
    ),
)


## 7. Aggregate Part III primary and sensitivity representations

Use maximum valid pre-landmark Part III scores across recorded medication states for the primary representation. This preserves the revision convention currently approved for manuscript continuity.

In [21]:
part_iii = tables["updrs_part_iii"]
part_iii_scores = ["NP3TOT", "NP3GAIT", "NP3PSTBL", "NHY"]
part_iii_flags = [
    "NP3GAIT_WAS_101",
    "NP3PSTBL_WAS_101",
    "NHY_WAS_101",
]

patient = add_features(
    patient,
    max_values(part_iii, part_iii_scores, "_COMBINED_MAX"),
)
patient = add_features(
    patient,
    max_values(part_iii, part_iii_flags, "_EVER"),
)


Aggregate ON and OFF examinations separately for the named sensitivity. These values do not replace the combined-state fields in the primary table.

In [22]:
state_text = part_iii["PDSTATE"].astype("string").str.upper()

for state in ["ON", "OFF"]:
    state_rows = part_iii.loc[state_text.str.startswith(state, na=False)]
    patient = add_features(
        patient,
        max_values(state_rows, part_iii_scores, f"_{state}_MAX"),
    )
    patient = add_features(
        patient,
        max_values(state_rows, part_iii_flags, f"_{state}_EVER"),
    )


## 8. Aggregate Part IV, clinical history, BMI, and orthostatic measures

Represent Part IV scores as historical burden and preserve positive histories for postural hypotension and REM-sleep behavior symptoms.

In [23]:
part_iv_fields = [
    "NP4TOT", "NP4WDYSK", "NP4DYSKI", "NP4OFF",
    "NP4FLCTI", "NP4FLCTX", "NP4DYSTN",
]
patient = add_features(
    patient,
    max_values(tables["updrs_part_iv"], part_iv_fields),
)
patient = add_features(
    patient,
    categorical_history(tables["other_clinical"], "FEATPOSHYP").to_frame(),
)
patient = add_features(
    patient,
    categorical_history(tables["other_clinical"], "FEATSUGRBD").to_frame(),
)


Use the latest valid cleaned BMI. Select all orthostatic measurements from one coherent visit before calculating within-visit standing-minus-supine changes.

In [24]:
vitals = tables["vital_signs"]
patient = add_features(
    patient,
    latest_values(vitals, ["BMI"], "latest status"),
)

orthostatic_fields = [
    "SYSSUP", "DIASUP", "HRSUP", "SYSSTND", "DIASTND", "HRSTND",
]
orthostatic = latest_row(
    vitals,
    orthostatic_fields,
    "orthostatic coherent visit",
)
orthostatic["SYS_CHANGE_STANDING_MINUS_SUPINE"] = (
    orthostatic["SYSSTND"] - orthostatic["SYSSUP"]
)
orthostatic["DIA_CHANGE_STANDING_MINUS_SUPINE"] = (
    orthostatic["DIASTND"] - orthostatic["DIASUP"]
)
orthostatic["HR_CHANGE_STANDING_MINUS_SUPINE"] = (
    orthostatic["HRSTND"] - orthostatic["HRSUP"]
)
patient = add_features(patient, orthostatic)


Retain the BMI history-quality flag for auditing and finish the 1,280-patient representation table.

In [25]:
patient = add_features(
    patient,
    vitals.groupby("PATNO")["HEIGHT_CONFLICT_UNRESOLVED"]
    .max()
    .rename("BMI_HEIGHT_HISTORY_UNRESOLVED")
    .to_frame(),
)
patient = patient.reset_index()

assert len(patient) == 1_280
assert patient["PATNO"].is_unique
print("Aggregated patient rows before modeling eligibility:", len(patient))


Aggregated patient rows before modeling eligibility: 1280


## 9. Declare the final 26-feature universe

Define the only primary modeling universe used in the final pipeline: 25 paper-aligned variables plus maximum eligible constipation (`NP1CNST`). The Neuro-QoL mobility questionnaire enters only through the revision's 8-item Gaussian score, as in the published model. Its seven individual items are not separate candidates because they would duplicate the score (decision D28). Later feature selection may retain a subset, but it must operate inside the training pipeline.


In [26]:
primary_features = [
    "Years_since_PD_diagnosis", "Age", "DXPOSINS", "DXRIGID", "DOPTHERST",
    "MCATOT", "FRZGT12M", "SCAU14", "SCAU16", "GDS_TOTAL",
    "NQ_GAUSSIAN_REVISION", "NP1RTOT", "BMI", "NP1SLPD", "NP1URIN",
    "NP3GAIT_COMBINED_MAX", "NP3PSTBL_COMBINED_MAX", "NHY_COMBINED_MAX",
    "FEATPOSHYP", "NP3TOT_COMBINED_MAX", "NP4TOT", "ANYFAMPD", "DXTREMOR",
    "DXBRADY", "DOMSIDE", "NP1CNST",
]

assert len(primary_features) == 26
assert len(set(primary_features)) == 26
print(f"Final candidate universe: {len(primary_features)} features")


Final candidate universe: 26 features


## 10. Apply the locked 1,040-patient primary eligibility rule

Retain a patient when at least one of the established longitudinal eligibility variables is recorded before the landmark. This list defines cohort eligibility only; it does not create another modeling feature set. Static age and diagnosis-history fields do not satisfy eligibility by themselves.


In [27]:
eligibility_longitudinal_features = [
    "DOPTHERST", "MCATOT", "FRZGT12M", "SCAU14", "SCAU16", "GDS_TOTAL",
    "NQ_GAUSSIAN_REVISION", "NP1RTOT", "BMI", "NP1SLPD", "NP1URIN",
    "NP3GAIT_COMBINED_MAX", "NP3PSTBL_COMBINED_MAX", "NHY_COMBINED_MAX",
    "FEATPOSHYP", "NP3TOT_COMBINED_MAX", "NP4TOT",
]

has_eligible_longitudinal_data = patient[
    eligibility_longitudinal_features
].notna().any(axis=1)
primary_ids = patient.loc[
    has_eligible_longitudinal_data,
    "PATNO",
]

assert len(primary_ids) == 1_040
print(f"Primary cohort: {len(primary_ids):,} patients")


Primary cohort: 1,040 patients


Derive the broader 1,043-patient cohort sensitivity using availability among the 25 established variables that preceded the new constipation candidate. The removed Neuro-QoL items share the Gaussian score's missingness, so the cohort is unchanged at 1,043. This changes cohort eligibility only; both cohorts use the same 26 predictor columns.


In [28]:
broader_eligibility_features = [
    feature for feature in primary_features
    if feature != "NP1CNST"
]
has_broader_eligible_data = patient[
    broader_eligibility_features
].notna().any(axis=1)
broader_ids = patient.loc[has_broader_eligible_data, "PATNO"]

assert len(broader_ids) == 1_043
assert set(primary_ids).issubset(set(broader_ids))
assert len(set(broader_ids).difference(primary_ids)) == 3

cohort_coverage = pd.DataFrame([
    {"definition": "Outcome-eligible spine", "patients": len(metadata)},
    {"definition": "Broader cohort sensitivity", "patients": len(broader_ids)},
    {"definition": "Primary cohort", "patients": len(primary_ids)},
    {"definition": "Additional patients in broader sensitivity", "patients": 3},
])

display(cohort_coverage)


,definition,patients
0,Outcome-eligible spine,1280
1,Broader cohort sensitivity,1043
2,Primary cohort,1040
3,Additional patients in broader sensitivity,3


## 11. Build the primary predictor and metadata tables

Align the primary patient rows and outcome metadata in the same deterministic patient order. Outcome and timing columns remain separate from predictors.

In [29]:
primary_patient = patient.loc[
    patient["PATNO"].isin(primary_ids)
].reset_index(drop=True)
primary_metadata = metadata.loc[
    metadata["PATNO"].isin(primary_ids)
].reset_index(drop=True)
primary_26 = primary_patient[["PATNO", *primary_features]].copy()

assert primary_patient["PATNO"].equals(primary_metadata["PATNO"])
print("Primary predictor shape:", primary_26.shape)


Primary predictor shape: (1040, 27)


## 12. Build one-change sensitivity inputs

Create the broader-cohort sensitivity using the same primary 26 predictors and a separately aligned metadata file.

In [30]:
broader_patient = patient.loc[
    patient["PATNO"].isin(broader_ids)
].reset_index(drop=True)
broader_metadata = metadata.loc[
    metadata["PATNO"].isin(broader_ids)
].reset_index(drop=True)
broader_26 = broader_patient[["PATNO", *primary_features]].copy()

assert broader_patient["PATNO"].equals(broader_metadata["PATNO"])


Create the latest-constipation sensitivity by changing only the constipation column while retaining its model-facing name.

In [31]:
sensitivity_latest_constipation = primary_26.copy()
sensitivity_latest_constipation["NP1CNST"] = primary_patient["NP1CNST_LATEST"]

shared_without_constipation = [
    column
    for column in primary_26.columns
    if column != "NP1CNST"
]
assert primary_26[shared_without_constipation].equals(
    sensitivity_latest_constipation[shared_without_constipation]
)


Create the `101`-flag sensitivity by adding only the three combined-history assessment-status flags to the primary 26 predictors.

In [32]:
combined_101_flags = [
    "NP3GAIT_WAS_101_EVER",
    "NP3PSTBL_WAS_101_EVER",
    "NHY_WAS_101_EVER",
]
sensitivity_101_flags = primary_26.join(
    primary_patient[combined_101_flags]
)

assert sensitivity_101_flags.shape[1] - 1 == 29


Create the ON/OFF sensitivity by replacing the four combined-state Part III fields with eight state-specific fields. Availability indicators will be derived from these missingness patterns inside each training fold.

In [33]:
combined_part_iii = [
    "NP3TOT_COMBINED_MAX",
    "NP3GAIT_COMBINED_MAX",
    "NP3PSTBL_COMBINED_MAX",
    "NHY_COMBINED_MAX",
]
state_specific_part_iii = [
    "NP3TOT_ON_MAX",
    "NP3GAIT_ON_MAX",
    "NP3PSTBL_ON_MAX",
    "NHY_ON_MAX",
    "NP3TOT_OFF_MAX",
    "NP3GAIT_OFF_MAX",
    "NP3PSTBL_OFF_MAX",
    "NHY_OFF_MAX",
]

on_off_base = primary_26.drop(columns=combined_part_iii)
sensitivity_on_off = on_off_base.join(
    primary_patient[state_specific_part_iii]
)

assert not set(combined_part_iii).intersection(sensitivity_on_off.columns)
assert set(state_specific_part_iii).issubset(sensitivity_on_off.columns)
assert sensitivity_on_off.shape[1] - 1 == 30


List every model-facing dataset and its role. Revision-compatible preprocessing uses the primary 26 table and therefore requires no additional patient table.

In [34]:
dataset_manifest = pd.DataFrame([
    {
        "dataset": "primary_1040_26_predictors",
        "patients": len(primary_26),
        "features": primary_26.shape[1] - 1,
        "role": "primary",
    },
    {
        "dataset": "sensitivity_1043_cohort",
        "patients": len(broader_26),
        "features": broader_26.shape[1] - 1,
        "role": "one-change sensitivity",
    },
    {
        "dataset": "sensitivity_latest_constipation",
        "patients": len(sensitivity_latest_constipation),
        "features": sensitivity_latest_constipation.shape[1] - 1,
        "role": "one-change sensitivity",
    },
    {
        "dataset": "sensitivity_part_iii_101_flags",
        "patients": len(sensitivity_101_flags),
        "features": sensitivity_101_flags.shape[1] - 1,
        "role": "one-change sensitivity",
    },
    {
        "dataset": "sensitivity_part_iii_on_off",
        "patients": len(sensitivity_on_off),
        "features": sensitivity_on_off.shape[1] - 1,
        "role": "one-change sensitivity",
    },
])

display(dataset_manifest)


,dataset,patients,features,role
0,primary_1040_26_predictors,1040,26,primary
1,sensitivity_1043_cohort,1043,26,one-change sensitivity
2,sensitivity_latest_constipation,1040,26,one-change sensitivity
3,sensitivity_part_iii_101_flags,1040,29,one-change sensitivity
4,sensitivity_part_iii_on_off,1040,30,one-change sensitivity


## 13. Summarize coverage and tie handling

Measure primary-feature availability without filling missing values. Also expose every deterministic same-month tie encountered by a latest-value rule.

In [35]:
feature_coverage = pd.DataFrame({
    "feature": primary_features,
    "recorded_patients": [
        primary_26[feature].notna().sum()
        for feature in primary_features
    ],
})
feature_coverage["recorded_percent"] = (
    100 * feature_coverage["recorded_patients"] / len(primary_26)
).round(1)

tie_audit = pd.DataFrame(selection_audits)

display(feature_coverage)
display(tie_audit)


,feature,recorded_patients,recorded_percent
0,Years_since_PD_diagnosis,1040,100.0
1,Age,1040,100.0
2,DXPOSINS,1039,99.9
3,DXRIGID,1040,100.0
4,DOPTHERST,838,80.6
5,MCATOT,1040,100.0
6,FRZGT12M,977,93.9
7,SCAU14,1040,100.0
8,SCAU16,1040,100.0
9,GDS_TOTAL,1040,100.0


,selection,patients_with_value,latest_month_ties,ties_with_different_values
0,demographics coherent row,1043,0,0
1,diagnosis coherent row,1043,0,0
2,latest status: MCATOT,1040,0,0
3,latest status: GDS_TOTAL,1040,0,0
4,latest-status alternative: NP1CNST,1040,0,0
5,Neuro-QoL coherent form,976,0,0
6,latest status: BMI,1040,0,0
7,orthostatic coherent visit,1043,2,0


Show the final class counts from the separate primary metadata file. This is a structural check, not feature analysis.

In [36]:
class_counts = (
    primary_metadata["falls_class"]
    .value_counts()
    .sort_index()
    .rename_axis("falls_class")
    .rename("patients")
    .reset_index()
)
class_counts["label"] = class_counts["falls_class"].map({
    0: "no fall",
    1: "rare fall",
    2: "recurrent fall",
})

display(class_counts[["falls_class", "label", "patients"]])


,falls_class,label,patients
0,0,no fall,712
1,1,rare fall,227
2,2,recurrent fall,101


## 14. Validate the final aggregation

Define a compact validation helper that records each decision and stops immediately on a failed scientific or structural invariant.

In [37]:
validation_rows = []


def check(name, condition, detail):
    passed = bool(condition)
    validation_rows.append({
        "check": name,
        "passed": passed,
        "detail": detail,
    })
    if not passed:
        raise AssertionError(f"{name}: {detail}")


Check primary cohort size, class composition, patient alignment, feature counts, and separation of outcomes from every predictor table.

In [38]:
primary_cohort_tables = [
    primary_26,
    sensitivity_latest_constipation,
    sensitivity_101_flags,
    sensitivity_on_off,
]

check("primary cohort size", len(primary_26) == 1_040, f"observed={len(primary_26)}")
check(
    "primary class counts",
    primary_metadata["falls_class"].value_counts().sort_index().to_dict()
    == {0: 712, 1: 227, 2: 101},
    "expected 712/227/101",
)
check(
    "unique patients",
    all(frame["PATNO"].is_unique for frame in [*primary_cohort_tables, broader_26]),
    "all primary and sensitivity predictor tables",
)
check(
    "primary patient alignment",
    all(frame["PATNO"].equals(primary_metadata["PATNO"]) for frame in primary_cohort_tables),
    "all 1,040-patient tables match primary metadata order",
)
check(
    "broader patient alignment",
    broader_26["PATNO"].equals(broader_metadata["PATNO"]),
    "1,043-patient predictors match broader metadata order",
)
check("primary feature count", primary_26.shape[1] - 1 == 26, "expected 26")


Check that outcomes and dates remain outside predictors and that missing predictor values remain available for later training-only preprocessing.

In [39]:
forbidden = {"falls_raw", "falls_class", "outcome_date", "index_date"}
check(
    "outcome separation",
    all(forbidden.isdisjoint(frame.columns) for frame in [*primary_cohort_tables, broader_26]),
    "no outcome or timing field appears in predictors",
)
check(
    "no imputation",
    primary_26.drop(columns="PATNO").isna().any().any(),
    "primary predictor missingness remains",
)
check(
    "broader sensitivity size",
    len(broader_26) == 1_043,
    f"observed={len(broader_26)}",
)
check(
    "broader sensitivity alignment",
    broader_26["PATNO"].equals(broader_metadata["PATNO"]),
    "predictors match broader metadata order",
)


Validate unable-to-rate handling, categorical meanings, BMI plausibility, and the mutually exclusive constipation definitions.

In [40]:
score_fields = [
    "NP3GAIT_COMBINED_MAX",
    "NP3PSTBL_COMBINED_MAX",
    "NHY_COMBINED_MAX",
    "NP3GAIT_ON_MAX",
    "NP3PSTBL_ON_MAX",
    "NHY_ON_MAX",
    "NP3GAIT_OFF_MAX",
    "NP3PSTBL_OFF_MAX",
    "NHY_OFF_MAX",
    "NP4WDYSK",
    "NP4DYSKI",
    "NP4OFF",
    "NP4FLCTI",
    "NP4FLCTX",
    "NP4DYSTN",
]
check(
    "101 removed from clinical scores",
    not primary_patient[score_fields].eq(101).any().any(),
    "all audited Part III and Part IV representations",
)
check(
    "valid postural-hypotension categories",
    set(primary_26["FEATPOSHYP"].dropna().unique())
    <= {"No", "Yes", "Uncertain"},
    "No/Yes/Uncertain only",
)
check(
    "valid cleaned BMI",
    primary_26["BMI"].dropna().between(10, 60).all(),
    "all recorded BMI values within 10–60",
)


Verify that the named sensitivities each change only the intended assumption and preserve valid flag values.

In [41]:
paired_constipation = pd.DataFrame({
    "maximum": primary_26["NP1CNST"],
    "latest": sensitivity_latest_constipation["NP1CNST"],
}).dropna()

check(
    "latest constipation changes one column",
    primary_26[shared_without_constipation].equals(
        sensitivity_latest_constipation[shared_without_constipation]
    ),
    "25 shared predictors are identical",
)
check(
    "maximum dominates latest constipation",
    paired_constipation["maximum"].ge(paired_constipation["latest"]).all(),
    "historical maximum cannot be below latest eligible value",
)
check(
    "101 flags valid",
    all(
        set(sensitivity_101_flags[field].dropna().unique()) <= {0, 1}
        for field in combined_101_flags
    ),
    "all three status flags are binary or missing",
)
check(
    "ON/OFF replaces combined state",
    not set(combined_part_iii).intersection(sensitivity_on_off.columns)
    and set(state_specific_part_iii).issubset(sensitivity_on_off.columns),
    "four combined fields replaced by eight state fields",
)


Collect the validation checks into one concise table. These checks use only the current notebook's inputs and declared rules.


In [42]:
validation = pd.DataFrame(validation_rows)
assert validation["passed"].all(), validation.loc[~validation["passed"]]
display(validation)
print(f"Validation checks passed: {validation['passed'].sum()}/{len(validation)}")


,check,passed,detail
0,primary cohort size,True,observed=1040
1,primary class counts,True,expected 712/227/101
2,unique patients,True,all primary and sensitivity predictor tables
3,primary patient alignment,True,"all 1,040-patient tables match primary metadat..."
4,broader patient alignment,True,"1,043-patient predictors match broader metadat..."
5,primary feature count,True,expected 26
6,outcome separation,True,no outcome or timing field appears in predictors
7,no imputation,True,primary predictor missingness remains
8,broader sensitivity size,True,observed=1043
9,broader sensitivity alignment,True,predictors match broader metadata order


Validation checks passed: 17/17


## 15. Save the final aggregation artifacts

Save the outcome-eligible spine, the single primary 26-feature input, and the approved sensitivity inputs. Predictor files contain no outcome or timing fields.


In [43]:
metadata.to_csv(
    output_directory / "outcome_eligible_patient_spine.csv",
    index=False,
)
primary_metadata.to_csv(
    output_directory / "primary_1040_outcome_metadata.csv",
    index=False,
)
primary_26.to_csv(
    output_directory / "primary_1040_26_predictors.csv",
    index=False,
)


Save each one-change sensitivity with a name that states exactly what differs from the primary analysis.

In [44]:
broader_metadata.to_csv(
    output_directory / "sensitivity_1043_outcome_metadata.csv",
    index=False,
)
broader_26.to_csv(
    output_directory / "sensitivity_1043_26_predictors.csv",
    index=False,
)
sensitivity_latest_constipation.to_csv(
    output_directory / "sensitivity_1040_latest_constipation_predictors.csv",
    index=False,
)
sensitivity_101_flags.to_csv(
    output_directory / "sensitivity_1040_part_iii_101_flags_predictors.csv",
    index=False,
)
sensitivity_on_off.to_csv(
    output_directory / "sensitivity_1040_part_iii_on_off_predictors.csv",
    index=False,
)


Save coverage, tie handling, dataset definitions, and self-contained validation results for downstream audit.


In [45]:
input_summary.to_csv(output_directory / "aggregation_input_summary.csv", index=False)
feature_coverage.to_csv(output_directory / "primary_feature_coverage.csv", index=False)
cohort_coverage.to_csv(output_directory / "cohort_coverage.csv", index=False)
class_counts.to_csv(output_directory / "primary_class_counts.csv", index=False)
dataset_manifest.to_csv(output_directory / "dataset_manifest.csv", index=False)
tie_audit.to_csv(output_directory / "same_month_tie_audit.csv", index=False)
validation.to_csv(output_directory / "aggregation_validation.csv", index=False)

print("Saved the final aggregation artifacts.")
print("No imputation, feature selection, or modeling was performed.")


Saved the final aggregation artifacts.
No imputation, feature selection, or modeling was performed.


## Result required before Notebook 03

The primary predictor file must contain 1,040 unique patients and exactly 26 candidate features. Its separately saved labels must contain 712 no-fall, 227 rare-fall, and 101 recurrent-fall patients. Every self-contained aggregation validation must pass.

Notebook 03 may then freeze the 20 paired 70/30 train/test splits and five inner folds. All missing predictor values remain untouched for training-only preprocessing.
